In [3]:
%pip install langchain langchain-groq langchain-community

     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ------------------------- -------------- 1.6/2.5 MB 33.6 MB/s eta 0:00:01
     ---------------------------------------- 2.5/2.5 MB 32.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/137.5 kB ? eta -:--:--
     ---------------------------------------- 137.5/137.5 kB ? eta 0:00:00
     ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
     ---------------------------------------- 1.0/1.0 MB 64.4 MB/s eta 0:00:00
     ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
     --------------------- ------------------ 1.2/2.1 MB 24.4 MB/s eta 0:00:01
     ---------------------------------------- 2.1/2.1 MB 27.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ------- -------------------------------- 2.3/12.9 MB 49.5 MB/s eta 0:00:01
     -------------- ------------------------- 4.6/12.9 MB 48.5 MB/s eta 0:00:01
     -------


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
import json
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일에서 환경 변수 로드

# LLM 초기화 (사용 가능한 모델 목록: mixtral-8x7b-32768, llama2-70b-4096, gemma-7b-it 등)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# ===== 1. StrOutputParser 사용 (문자열 파싱) =====
str_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_template("{topic}의 장점은 무엇인가요?")
chain_str = prompt | llm | str_parser

# ===== 2. JsonOutputParser 사용 (JSON 파싱) =====
json_parser = JsonOutputParser()
prompt_json = ChatPromptTemplate.from_template(
    """{topic}에 대해 JSON 형식으로 답변해줘:
    {{"장점": [], "단점": []}}"""
)
chain_json = prompt_json | llm | json_parser

# ===== 3. RunnableLambda로 커스텀 파서 =====
def custom_parser(response):
    """응답에서 메시지 내용만 추출"""
    return response.content.split('\n')[0]  # 첫 줄만 추출

custom_parser_chain = prompt | llm | RunnableLambda(custom_parser)

In [9]:
# ===== 체인 실행 예제 =====
topic = "Python"

# 1. 문자열 파서로 실행
print("=== StrOutputParser ===")
result_str = chain_str.invoke({"topic": topic})
print(f"타입: {type(result_str)}")
print(f"결과: {result_str[:100]}...")

# 2. JSON 파서로 실행
print("\n=== JsonOutputParser ===")
result_json = chain_json.invoke({"topic": topic})
print(f"타입: {type(result_json)}")
print(f"결과: {result_json}")

# 3. 커스텀 파서로 실행
print("\n=== Custom Parser ===")
result_custom = custom_parser_chain.invoke({"topic": topic})
print(f"타입: {type(result_custom)}")
print(f"결과: {result_custom}")

=== StrOutputParser ===
타입: <class 'langchain_core.messages.base.TextAccessor'>
결과: Python은 다양한 분야에서 사용되는 고급 프로그래밍 언어입니다. Python의 주요 장점은 다음과 같습니다.

1. **간단하고 읽기 쉬운 코드**: Python은 간결하고 명...

=== JsonOutputParser ===


OutputParserException: Invalid json output: **장점**

*   **간결하고 읽기 쉬운 코드**: Python은 간결하고 읽기 쉬운 코드를 작성할 수 있는 언어입니다. 이는 개발자가 코드를 더 빠르게 작성하고 유지보수할 수 있도록 합니다.
*   **빠른 개발**: Python은 빠르게 개발할 수 있는 언어입니다. 이는 개발자가 프로젝트를 더 빠르게 완료할 수 있도록 합니다.
*   **대화형 셸**: Python은 대화형 셸을 제공합니다. 이는 개발자가 코드를 즉시 실행하고 결과를 확인할 수 있도록 합니다.
*   **많은 라이브러리**: Python에는 많은 라이브러리가 있습니다. 이는 개발자가 다양한 기능을 쉽게 구현할 수 있도록 합니다.
*   **다양한 플랫폼 지원**: Python은 다양한 플랫폼을 지원합니다. 이는 개발자가 Windows, macOS, Linux 등 다양한 운영체제에서 코드를 실행할 수 있도록 합니다.

**단점**

*   **성능**: Python은 성능이 좋지 않은 언어입니다. 이는 대규모 프로젝트에서 성능 문제가 발생할 수 있습니다.
*   **보안**: Python은 보안이 좋지 않은 언어입니다. 이는 악성 코드가 쉽게 실행될 수 있습니다.
*   **학습 곡선**: Python은 학습 곡선이 높습니다. 이는 개발자가 Python을 배우기 위해 많은 시간과 노력을 들여야 합니다.
*   **라이브러리 관리**: Python의 라이브러리 관리가 어렵습니다. 이는 개발자가 라이브러리를 관리하기 위해 많은 시간과 노력을 들여야 합니다.
*   **멀티스레딩**: Python은 멀티스레딩이 어렵습니다. 이는 개발자가 멀티스레딩을 구현하기 위해 많은 시간과 노력을 들여야 합니다.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [13]:
extract_question = RunnableLambda(lambda x: f"질문: {x['question']}")
extract_context_len = RunnableLambda(lambda x: f"컨텍스트 길이: {len(x['context'])}")
extract_language = RunnableLambda(lambda x: f"질문 작성 언어: {x['lang']}")

parallel = RunnableParallel(
    question = extract_question,
    context_len = extract_context_len,
    language = extract_language
)

res = parallel.invoke({
    "question": "머신러닝은 무엇인가요",
    "context": "머신러닝은 인공지능의 한 분야로, 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터를 통해 학습할 수 있도록 하는 기술입니다.",
    "lang": "korean"
})

print(f"question: {res['question']}")
print(f"context_len: {res['context_len']}")
print(f"language: {res['language']}")

question: 질문: 머신러닝은 무엇인가요
context_len: 컨텍스트 길이: 69
language: 질문 작성 언어: korean


In [15]:

parallel = RunnableParallel(
    question = RunnableLambda(lambda x: x['question']),
    context_len = RunnableLambda(lambda x: len(x['context'])),
    language = RunnableLambda(lambda x: x['lang'])
)
merge_fn = RunnableLambda(
    lambda x: {
        "question": x['question'],
        "context_len": x['context_len'],
        "language": x['language']
    }
)

answer_prompt = ChatPromptTemplate.from_template(
    "{question}에 대해 {language}로 답변해줘. 컨텍스트 길이는 {context_len}이야."
)
parser = StrOutputParser()
chain = parallel | merge_fn | answer_prompt | llm | parser

ans = chain.invoke({
    "question": "머신러닝은 무엇인가요",
    "context": "머신러닝은 인공지능의 한 분야로, 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터를 통해 학습할 수 있도록 하는 기술입니다.",
    "lang": "일본어"
})

print(ans)

머신러닝(Machine Learning)은 컴퓨터가 데이터를 분석하고 학습하여 특정 패턴이나 규칙을 발견하는 기술입니다. 머신러닝은 컴퓨터가 데이터를 처리하고 분석하는 능력을 향상시키기 위해 사용됩니다. 머신러닝은 다음과 같은 방법으로 데이터를 처리합니다.

1. 데이터 수집: 머신러닝은 데이터를 수집하고 저장하는 과정을 거칩니다.
2. 데이터 전처리: 수집된 데이터를 정리하고 가공하는 과정을 거칩니다.
3. 모델 학습: 가공된 데이터를 사용하여 머신러닝 모델을 학습시키는 과정을 거칩니다.
4. 모델 평가: 학습된 모델을 평가하여 성능을 측정하는 과정을 거칩니다.
5. 모델 최적화: 모델의 성능을 향상시키기 위해 모델을 최적화하는 과정을 거칩니다.

머신러닝은 다양한 분야에서 사용됩니다. 예를 들어, 이미지 분류, 자연어 처리, 추천 시스템, 예측 모델링 등이 있습니다. 머신러닝은 컴퓨터가 데이터를 처리하고 분석하는 능력을 향상시키기 때문에 많은 분야에서 사용됩니다.


In [17]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough.assign(
    length = RunnableLambda(lambda x: len(x['text'])),
    uppercase = RunnableLambda(lambda x: x['text'].upper())
)

res=passthrough.invoke({"text": "Hello, World!"})

print(f"Length: {res['length']}")
print(f"Uppercase: {res['uppercase']}")
print(f"Original: {res['text']}")

Length: 13
Uppercase: HELLO, WORLD!
Original: Hello, World!


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

chain = RunnableParallel(
    input= RunnablePassthrough(),
    process= RunnableLambda(lambda x: x['text'].upper())
)

res = chain.invoke({"text": "Hello, World!"})
print(res)

{'input': {'text': 'Hello, World!'}, 'process': 'HELLO, WORLD!'}


: 